#**Лабораторная 1. Решите следующие задачи для данных велопарковок Сан-Франциско**
##**Задание**

*   Найти велосипед с максимальным временем пробега.
*   Найти наибольшее геодезическое расстояние между станциями.
*   Найти путь велосипеда с максимальным временем пробега через станции.
*   Найти количество велосипедов в системе.
*   Найти пользователей потративших на поездки более 3 часов.

##1. Устанавливаем PySpark

In [48]:
!pip -q install pyspark

##2. Импорты

In [58]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum
from pyspark.sql.functions import udf, col
from pyspark.sql.types import FloatType
from geopy.distance import geodesic
from pyspark.sql import functions as F

##3. Загрузка

In [50]:
spark = (
    SparkSession.builder
    .appName("Лабораторная 2")
    .master("local[*]")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)

Spark version: 4.0.2


##4. Чтение датасетов

In [51]:
data_trips = spark.read.format('csv').option('header', 'true').load('/content/trips.csv')
data_stations = spark.read.format('csv').option('header', 'true').load('/content/stations.csv')

data_trips.show(10)
data_stations.show(10)

+----+--------+---------------+--------------------+----------------+---------------+--------------------+--------------+-------+-----------------+--------+
|  id|duration|     start_date|  start_station_name|start_station_id|       end_date|    end_station_name|end_station_id|bike_id|subscription_type|zip_code|
+----+--------+---------------+--------------------+----------------+---------------+--------------------+--------------+-------+-----------------+--------+
|4576|      63|           NULL|South Van Ness at...|              66|8/29/2013 14:14|South Van Ness at...|            66|    520|       Subscriber|   94127|
|4607|    NULL|8/29/2013 14:42|  San Jose City Hall|              10|8/29/2013 14:43|  San Jose City Hall|            10|    661|       Subscriber|   95138|
|4130|      71|8/29/2013 10:16|Mountain View Cit...|              27|8/29/2013 10:17|Mountain View Cit...|            27|     48|       Subscriber|   97214|
|4251|      77|8/29/2013 11:29|  San Jose City Hall|      

##5. Найти велосипед с максимальным временем пробега

In [52]:
bike_totals = data_trips.groupBy("bike_id").agg(
    spark_sum(col("duration").cast("int")).alias("total_duration")
)

max_bike = bike_totals.orderBy(col("total_duration").desc()).limit(1)
max_bike.show()

+-------+--------------+
|bike_id|total_duration|
+-------+--------------+
|    535|      18611693|
+-------+--------------+



##6. Найти наибольшее геодезическое расстояние между станциями

In [53]:
def calc_distance(lat1, lon1, lat2, lon2):
    try:
        return geodesic((float(lat1), float(lon1)), (float(lat2), float(lon2))).kilometers
    except:
        return 0.0

distance_udf = udf(calc_distance, FloatType())

stations1 = data_stations.selectExpr("id as id1", "name as name1", "lat as lat1", "long as long1")
stations2 = data_stations.selectExpr("id as id2", "name as name2", "lat as lat2", "long as long2")

pairs = stations1.crossJoin(stations2).filter(col("id1") < col("id2"))

pairs_with_distance = pairs.withColumn(
    "distance_km",
    distance_udf(col("lat1"), col("long1"), col("lat2"), col("long2"))
)

max_result = pairs_with_distance.orderBy(col("distance_km").desc()).limit(1)
max_result.show(truncate=False)

+---+--------------------------+------------------+-----------+---+----------------------+--------+-------------------+-----------+
|id1|name1                     |lat1              |long1      |id2|name2                 |lat2    |long2              |distance_km|
+---+--------------------------+------------------+-----------+---+----------------------+--------+-------------------+-----------+
|16 |SJSU - San Salvador at 9th|37.333954999999996|-121.877349|60 |Embarcadero at Sansome|37.80477|-122.40323400000001|69.92097   |
+---+--------------------------+------------------+-----------+---+----------------------+--------+-------------------+-----------+



##7. Найти путь велосипеда с максимальным временем пробега через станции

In [54]:
bike_routes = data_trips.groupBy("bike_id").agg(
    F.sum(F.col("duration").cast("int")).alias("total_duration"),
    F.collect_list(
        F.struct("start_date", "start_station_name", "end_station_name")
    ).alias("trips")
)

max_bike = bike_routes.orderBy(F.col("total_duration").desc()).limit(1)
max_bike.show(truncate=False)

+-------+--------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

##8. Найти количество велосипедов в системе

In [55]:
num_bikes = data_trips.select("bike_id").distinct().count()
print(f"Количество велосипедов: {num_bikes}")

Количество велосипедов: 700


##9. Найти пользователей потративших на поездки более 3 часов

In [56]:
zip_totals = data_trips.groupBy("zip_code").agg(
    (spark_sum(col("duration").cast("int")) / 3600).alias("total_hours")
)

users_time = zip_totals.filter(col("total_hours") > 3)
users_time.show()

+--------+------------------+
|zip_code|       total_hours|
+--------+------------------+
|   94102| 5313.339166666667|
|   95134|202.22861111111112|
|   84606|26.429166666666667|
|   80305|50.251666666666665|
|   60070| 8.033055555555556|
|   95519|            8.4175|
|   43085|3.2416666666666667|
|   91910|14.024444444444445|
|   77339|3.8091666666666666|
|   48063|3.8208333333333333|
|   85022| 3.522777777777778|
|    1090| 5.664166666666667|
|    2136| 4.447222222222222|
|   11722| 6.758611111111111|
|   95138|           43.1375|
|   94610|1008.5077777777777|
|   94404| 997.0416666666666|
|   80301| 42.27472222222222|
|   91326|18.301388888888887|
|   90742|3.0458333333333334|
+--------+------------------+
only showing top 20 rows


##10. Останавливаем SparkSession

In [57]:
spark.stop()